# Step 5 — DICOM cache build

**Settings: Accelerator = `None` (CPU), Internet = `Off`.** Attach the competition data.

This is the single highest-leverage thing you can do for the image half of the competition, and it costs **zero GPU quota**.

## Why

Decoding DICOMs inside a training loop means your GPU sits idle while the CPU decompresses JPEG2000. At 24,371 series and ~30 slices each, that's most of your training time spent not training.

So we decode **once**, on CPU, and write a fixed-shape uint8 array. Every future training run then reads pre-decoded pixels straight from disk.

## What gets built

`cache.npy`, shape `(4407, 3, 24, 160, 160)` uint8 — about 8 GB.

Three series per study (one sagittal, one coronal, one axial), 24 slices each, resampled to 160x160.

## The four things this gets right

1. **Slice ordering by geometry.** Filenames and `InstanceNumber` are not reliably in anatomical order. We compute the slice normal from `ImageOrientationPatient` and sort by projecting `ImagePositionPatient` onto it. Get this wrong and your sequence model learns from shuffled anatomy.
2. **Per-series percentile normalisation.** MRI has no absolute scale like CT's Hounsfield units, and `WindowCenter` in this dataset ranges from 148 to 1660 — sometimes plain wrong (`ContrastBolusAgent` contained the value `LEFT`). We clip to the 1st–99th percentile of each series and rescale.
3. **Orientation canonicalisation.** Volume axes are flipped so each points consistently in patient space, then left knees are mirrored to look like right knees. Without this, `Medial` and `Lateral` mean opposite things depending on which leg was scanned — silently capping four of your twelve labels.
4. **Resumability.** A done-flag array means a crash costs you minutes, not the whole run.

### 1 · Setup

In [ ]:
import os, glob, time, warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, pydicom
from multiprocessing import Pool

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT  = "/kaggle/working"

PX      = 160     # in-plane resolution
NSLICE  = 24      # slices per series after resampling
NSERIES = 3       # sagittal, coronal, axial

train = pd.read_csv(f"{ROOT}/train.csv")
ser   = pd.read_csv(f"{ROOT}/train_series.csv")
studies = train["StudyInstanceUID"].tolist()
sidx = {s: i for i, s in enumerate(studies)}

print(f"studies {len(studies):,} | series {len(ser):,}")
print(f"cache shape ({len(studies)}, {NSERIES}, {NSLICE}, {PX}, {PX}) uint8"
      f" = {len(studies)*NSERIES*NSLICE*PX*PX/1e9:.1f} GB")
print("cores:", os.cpu_count())

### 2 · Choose which series to keep

Each study has ~5.5 series but we keep 3. `train_series.csv` gives plane, fluid-sensitivity and fat-suppression for free, so selection is metadata-driven rather than guesswork.

Within each plane we prefer **fluid-sensitive** sequences — those are where effusion, synovitis, contusion and Baker's cysts are visible. Fat-suppressed next, then whichever series has the most slices.

In [ ]:
ser["n"] = 1
cnt = ser.groupby(["StudyInstanceUID", "SeriesInstanceUID"]).size()

def pick(g):
    out = []
    for plane in ["Sagittal", "Coronal", "Axial"]:
        sub = g[g["Anatomical_Plane"] == plane]
        if len(sub) == 0:
            out.append(None); continue
        sub = sub.sort_values(["Fluid_Sensitive", "Fat_Suppression"], ascending=False)
        out.append(sub.iloc[0]["SeriesInstanceUID"])
    return out

t0 = time.time()
chosen = {}
for suid, g in ser.groupby("StudyInstanceUID"):
    chosen[suid] = pick(g)
print(f"selected in {time.time()-t0:.0f}s")

have = np.array([[c is not None for c in chosen.get(s, [None]*3)] for s in studies])
print(f"studies with sagittal {have[:,0].mean():.1%} | coronal {have[:,1].mean():.1%} | axial {have[:,2].mean():.1%}")
print(f"studies with all three: {have.all(1).mean():.1%}")
print(f"studies with none     : {(~have.any(1)).mean():.1%}")

### 3 · The decoder

**Geometric slice ordering.** `ImageOrientationPatient` gives two direction cosines — the image row and column directions in patient space. Their cross product is the slice normal. Projecting each slice's `ImagePositionPatient` onto that normal gives its true anatomical position, which is what we sort by.

**Canonical orientation.** For each volume axis we find which patient axis it most aligns with and flip it to a fixed sign. Then, for a left knee, we mirror the axis that runs medial–lateral. Note this is *not* always the in-plane horizontal: on sagittal images the medial–lateral direction is the through-slice axis, so the fix is reversing slice order, not flipping the picture.

In [ ]:
def load_series(dirpath, laterality_hint=None):
    files = os.listdir(dirpath)
    if not files:
        return None
    metas = []
    for f in files:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, f), stop_before_pixels=True)
            iop = getattr(d, "ImageOrientationPatient", None)
            ipp = getattr(d, "ImagePositionPatient", None)
            metas.append((f, np.array(iop, float) if iop else None,
                             np.array(ipp, float) if ipp else None,
                             getattr(d, "InstanceNumber", 0),
                             str(getattr(d, "Laterality", "") or "")))
        except Exception:
            continue
    if not metas:
        return None

    iops = [m[1] for m in metas if m[1] is not None]
    if iops:
        iop = iops[0]
        r, c = iop[:3], iop[3:]
        n = np.cross(r, c)
        pos = [(float(np.dot(m[2], n)) if m[2] is not None else float(m[3])) for m in metas]
    else:
        r, c, n = np.array([1.,0,0]), np.array([0,1.,0]), np.array([0,0,1.])
        pos = [float(m[3]) for m in metas]

    order = np.argsort(pos)
    metas = [metas[i] for i in order]

    # resample to NSLICE evenly spaced
    take = np.linspace(0, len(metas)-1, NSLICE).round().astype(int)

    imgs = []
    for i in take:
        try:
            d = pydicom.dcmread(os.path.join(dirpath, metas[i][0]))
            a = d.pixel_array.astype(np.float32)
            if hasattr(d, "RescaleSlope"):
                a = a * float(d.RescaleSlope) + float(getattr(d, "RescaleIntercept", 0))
        except Exception:
            a = np.zeros((PX, PX), np.float32)
        imgs.append(cv2.resize(a, (PX, PX), interpolation=cv2.INTER_AREA))
    vol = np.stack(imgs)                                   # (Z, Y, X)

    # per-SERIES percentile normalisation (MRI has no absolute scale)
    lo, hi = np.percentile(vol, [1, 99])
    if hi <= lo: hi = lo + 1
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    # canonical orientation: flip each axis to a fixed patient-space sign
    axes = [n, c, r]                                       # vol axes Z, Y, X
    for ax, vec in enumerate(axes):
        dom = int(np.argmax(np.abs(vec)))
        if vec[dom] < 0:
            vol = np.flip(vol, axis=ax)
        axes[ax] = vec * (1 if vec[dom] >= 0 else -1)

    # mirror LEFT knees onto the RIGHT convention.
    # patient x-axis is L-R in DICOM LPS; find whichever volume axis tracks it.
    lat = laterality_hint or next((m[4] for m in metas if m[4]), "")
    if str(lat).upper().startswith("L"):
        lr_axis = int(np.argmax([abs(v[0]) for v in axes]))
        vol = np.flip(vol, axis=lr_axis)

    return (vol * 255).astype(np.uint8)

print("decoder ready")

### 4 · Benchmark on 8 studies

Measures throughput and projects total runtime before you commit to the full job.

In [ ]:
def study_paths(suid):
    return [(f"{ROOT}/train_series/{suid}/{c}" if c else None) for c in chosen.get(suid, [None]*3)]

t0 = time.time(); ok = 0
for s in studies[:8]:
    for p in study_paths(s):
        if p and os.path.isdir(p):
            v = load_series(p)
            if v is not None: ok += 1
el = time.time() - t0
per = el / 8
print(f"{el:.1f}s for 8 studies -> {per:.2f}s/study ({ok} series decoded)")
print(f"projected 1 core : {per*len(studies)/3600:.1f} h")
print(f"projected 4 cores: {per*len(studies)/4/3600:.1f} h")

### 5 · Build the cache

Workers write directly into a shared memory-mapped array at their own row offsets — no 8 GB of data passing through inter-process pipes.

`done.npy` tracks completed studies, so re-running resumes rather than restarting.

In [ ]:
CACHE = f"{OUT}/cache.npy"
DONE  = f"{OUT}/done.npy"
SHAPE = (len(studies), NSERIES, NSLICE, PX, PX)

if not os.path.exists(CACHE):
    np.lib.format.open_memmap(CACHE, mode="w+", dtype=np.uint8, shape=SHAPE)
    np.save(DONE, np.zeros(len(studies), np.uint8))
    print("created", CACHE)

done = np.load(DONE)
print(f"already done: {done.sum():,}/{len(studies):,}")


def work(i):
    suid = studies[i]
    arr = np.lib.format.open_memmap(CACHE, mode="r+")
    got = 0
    for k, p in enumerate(study_paths(suid)):
        if p and os.path.isdir(p):
            try:
                v = load_series(p)
                if v is not None:
                    arr[i, k] = v; got += 1
            except Exception:
                pass
    del arr
    return i, got


todo = [i for i in range(len(studies)) if not done[i]]
print(f"to process: {len(todo):,}\n")

t0 = time.time()
with Pool(4) as pool:
    for n, (i, got) in enumerate(pool.imap_unordered(work, todo, chunksize=8), 1):
        done[i] = 1
        if n % 200 == 0 or n == len(todo):
            np.save(DONE, done)
            el = time.time() - t0
            print(f"  {n}/{len(todo)}  {el/n:.2f}s/study  eta {(len(todo)-n)*el/n/60:.0f} min")
np.save(DONE, done)
print(f"\ncomplete: {done.sum():,}/{len(studies):,} in {(time.time()-t0)/60:.0f} min")

### 6 · Verify

A cache that silently contains zeros is worse than no cache — you would train on black images and never know.

In [ ]:
arr = np.load(CACHE, mmap_mode="r")
rng = np.random.default_rng(0)
chk = rng.choice(len(studies), 300, replace=False)

nonzero = np.array([[arr[i, k].max() > 0 for k in range(NSERIES)] for i in chk])
print(f"series slots populated: {nonzero.mean():.1%}")
print(f"  sagittal {nonzero[:,0].mean():.1%} | coronal {nonzero[:,1].mean():.1%} | axial {nonzero[:,2].mean():.1%}")
print(f"studies with >=1 series: {nonzero.any(1).mean():.1%}")
print(f"studies completely empty: {(~nonzero.any(1)).mean():.1%}")

sub = arr[chk[:50]].astype(np.float32)
sub = sub[sub.max(axis=(2,3)) > 0] if sub.ndim == 5 else sub
print(f"\nintensity  mean {arr[chk[:50]].mean():.1f}  std {arr[chk[:50]].std():.1f}  (expect ~60-110 / ~50-80)")

pd.DataFrame({"StudyInstanceUID": studies, "done": done}).to_csv(f"{OUT}/cache_index.csv", index=False)
print("\nwrote cache_index.csv")

### 7 · Look at it

Always eyeball medical imaging output. Bugs in slice ordering or normalisation are obvious to an eye and invisible in a summary statistic.

In [ ]:
import matplotlib.pyplot as plt
i = int(np.where(done > 0)[0][0])
fig, ax = plt.subplots(3, 8, figsize=(18, 7))
for k in range(3):
    for j in range(8):
        ax[k, j].imshow(arr[i, k, j*3], cmap="gray", vmin=0, vmax=255)
        ax[k, j].axis("off")
    ax[k, 0].set_title(["sagittal", "coronal", "axial"][k], loc="left", fontsize=11)
plt.suptitle("one study - every 3rd slice - should progress smoothly through the joint")
plt.tight_layout(); plt.show()